# Create KEGG reference files

1. REF_KEGG2LABEL: dictionary of KEGG to label
2. REF_KEGG2FORMULA: dictionary of KEGG to shortened formula
3. REF_KEGG2NAMES: dictionary of KEGG to all synonyms

In [ ]:
import re
import lzma
import json
import glob

import os
import requests
import time

# Data

## Download the reactions, enzymes, pathways, and reaction links from KEGG

Data was obtained on July 22, 2025 from [www.kegg.jp](www.kegg.jp)

In [ ]:
# Downloads various types of data from KEGG API and organizes them into separate folders

# --- CONFIGURATION ---
BASE_URL = "https://rest.kegg.jp"
BASE_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)))
DELAY = 1.5  # seconds between requests, be polite to the KEGG API

# Define subdirectories for different data types
DIRS = {
    'reactions': os.path.join(BASE_DIR, 'reactions'),
    'enzymes': os.path.join(BASE_DIR, 'enzymes'),
    'pathways': os.path.join(BASE_DIR, 'pathways'),
    'links': os.path.join(BASE_DIR, 'links')
}

# Create all required directories
for directory in DIRS.values():
    os.makedirs(directory, exist_ok=True)

# --- HELPER FUNCTIONS ---
def fetch_data(endpoint, prefix=""):
    """
    Fetch data from KEGG API
    
    Args:
        endpoint: API endpoint to fetch
        prefix: Optional URL prefix
        
    Returns:
        Response text if successful, None otherwise
    """
    url = f"{BASE_URL}{prefix}{endpoint}"
    print(f"Fetching {url}...")
    
    r = requests.get(url)
    if r.status_code == 200:
        return r.text
    else:
        print(f"❌ Failed to fetch {url}: {r.status_code}")
        return None

def save_file(content, directory, filename):
    """
    Save content to a file in the specified directory
    
    Args:
        content: Content to save
        directory: Directory to save to
        filename: Name of the file
    """
    filepath = os.path.join(directory, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
    return filepath

def fetch_and_save(endpoint, directory, filename, prefix=""):
    """
    Fetch data from KEGG API and save it to a file
    
    Args:
        endpoint: API endpoint to fetch
        directory: Directory to save to
        filename: Name of the file
        prefix: Optional URL prefix
        
    Returns:
        Response text if successful, None otherwise
    """
    data = fetch_data(endpoint, prefix)
    if data:
        save_file(data, directory, filename)
    return data

def parse_entry_ids(text, prefix_to_remove=None):
    """
    Parse entry IDs from KEGG list response
    
    Args:
        text: Response text from KEGG API
        prefix_to_remove: Prefix to remove from IDs (e.g., 'rn:', 'ec:')
        
    Returns:
        List of entry IDs
    """
    lines = text.strip().split("\n")
    if prefix_to_remove:
        return [line.split()[0].replace(prefix_to_remove, "") for line in lines]
    else:
        return [line.split()[0] for line in lines]

# --- REACTION DOWNLOAD FUNCTIONS ---
def download_all_reaction_ids():
    """
    Download all reaction IDs from KEGG
    
    Returns:
        List of reaction IDs
    """
    data = fetch_and_save("/list/reaction", DIRS['reactions'], "all_reaction_ids.txt")
    if not data:
        raise Exception("Failed to fetch reaction list")
    
    rxn_ids = parse_entry_ids(data, "rn:")
    return rxn_ids

def download_reaction_entries(rxn_ids):
    """
    Download individual reaction entries
    
    Args:
        rxn_ids: List of reaction IDs to download
    """
    for rxn in rxn_ids:
        out_path = os.path.join(DIRS['reactions'], f"reaction_{rxn}.txt")
        
        # Skip if already downloaded
        if os.path.exists(out_path):
            continue
            
        fetch_and_save(f"/get/rn:{rxn}", DIRS['reactions'], f"reaction_{rxn}.txt")
        time.sleep(DELAY)

# --- ENZYME DOWNLOAD FUNCTIONS ---
def download_all_enzyme_ids():
    """
    Download all enzyme IDs from KEGG
    
    Returns:
        List of enzyme IDs (EC numbers)
    """
    data = fetch_and_save("/list/enzyme", DIRS['enzymes'], "all_enzyme_ids.txt")
    if not data:
        raise Exception("Failed to fetch enzyme list")
    
    ec_numbers = parse_entry_ids(data, "ec:")
        
    return ec_numbers

def download_enzyme_data(ec_numbers):
    """
    Download enzyme data and related information
    
    Args:
        ec_numbers: List of EC numbers to download
    """
    for ec in ec_numbers:
        print(f"\nProcessing EC:{ec}")
        
        # Download enzyme entry
        fetch_and_save(f"/get/ec:{ec}", DIRS['enzymes'], f"enzyme_ec_{ec}.txt")
        
        # Download linked reactions
        fetch_and_save(f"/link/reaction/ec:{ec}", DIRS['links'], f"reaction_links_ec_{ec}.txt")

        # Download linked pathways
        fetch_and_save(f"/link/pathway/ec:{ec}", DIRS['links'], f"pathway_links_ec_{ec}.txt")
        
        time.sleep(DELAY)

# --- MAIN EXECUTION ---
def download_reactions():
    """Download all reaction data"""
    print("\n=== Downloading KEGG Reactions ===")
    rxn_ids = download_all_reaction_ids()
    download_reaction_entries(rxn_ids)
    print(f"✅ Downloaded {len(rxn_ids)} reactions")

def download_enzymes():
    """Download all enzyme data"""
    print("\n=== Downloading KEGG Enzymes ===")
    ec_numbers = download_all_enzyme_ids()
    download_enzyme_data(ec_numbers)
    print(f"✅ Downloaded {len(ec_numbers)} enzymes")


NameError: name '__file__' is not defined

In [ ]:
# if __name__ == "__main__":
print("Starting KEGG data download...")

# Download reactions
download_reactions()

# Download enzymes
download_enzymes()

print("\n✅ KEGG data download complete.")

## Download KEGG compounds

In [ ]:
import requests
import time
from pathlib import Path

BASE_URL = "http://rest.kegg.jp"
OUTPUT_DIR = Path("kegg_compound_flats")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def fetch_compound_ids():
    """
    Fetch the list of all KEGG compound IDs using the REST API.
    """
    url = f"{BASE_URL}/list/compound"
    print("Fetching compound list...")
    resp = requests.get(url)
    resp.raise_for_status()
    lines = resp.text.strip().split('\n')
    compound_ids = [line.split('\t')[0] for line in lines]
    print(f"Retrieved {len(compound_ids)} compound IDs")
    return compound_ids

def fetch_compound_entry(cid: str):
    """
    Fetch the full entry for a given KEGG compound ID.
    """
    url = f"{BASE_URL}/get/{cid}"
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.text

def main():
    compound_ids = fetch_compound_ids()

    for cid in compound_ids:
        try:
            entry = fetch_compound_entry(cid)
            file_path = OUTPUT_DIR / f"{cid}.txt"
            with open(file_path, 'w') as f:
                f.write(entry)

            print(f"Saved {cid} → {file_path}")
            time.sleep(1)  # modest delay to respect server load

        except Exception as e:
            print(f"Error retrieving {cid}: {e}")
            continue

main()


Fetching compound list...
Retrieved 19513 compound IDs
Saved C00001 → kegg_compound_flats\C00001.txt
Saved C00002 → kegg_compound_flats\C00002.txt
Saved C00003 → kegg_compound_flats\C00003.txt
Saved C00004 → kegg_compound_flats\C00004.txt
Saved C00005 → kegg_compound_flats\C00005.txt
Saved C00006 → kegg_compound_flats\C00006.txt
Saved C00007 → kegg_compound_flats\C00007.txt
Saved C00008 → kegg_compound_flats\C00008.txt
Saved C00009 → kegg_compound_flats\C00009.txt
Saved C00010 → kegg_compound_flats\C00010.txt
Saved C00011 → kegg_compound_flats\C00011.txt
Saved C00012 → kegg_compound_flats\C00012.txt
Saved C00013 → kegg_compound_flats\C00013.txt
Saved C00014 → kegg_compound_flats\C00014.txt
Saved C00015 → kegg_compound_flats\C00015.txt
Saved C00016 → kegg_compound_flats\C00016.txt
Saved C00017 → kegg_compound_flats\C00017.txt
Saved C00018 → kegg_compound_flats\C00018.txt
Saved C00019 → kegg_compound_flats\C00019.txt
Saved C00020 → kegg_compound_flats\C00020.txt
Saved C00021 → kegg_compo

## Download and save ChEBI to KEGG compound mappings

In [1]:
import gzip
from rdflib import Graph, Namespace
import csv, json
from tqdm import tqdm

# Load ontology file directly from gzip
g = Graph()
with gzip.open("chebi.owl.gz", "rb") as f:
    g.parse(f, format="xml")

# Namespaces
OBOINOWL = Namespace("http://www.geneontology.org/formats/oboInOwl#")
# 4min 30s

In [2]:
mapping = {}

for s, p, o in g.triples((None, OBOINOWL.hasDbXref, None)):
    xref = str(o)  # convert Literal → string
    if xref.startswith("KEGG:"):
        kegg_id = xref.split(":")[-1]
        if kegg_id.startswith('C'):
            chebi_id = s.split("/")[-1].replace("_", ":")
            mapping.setdefault(chebi_id, []).append(kegg_id)

print(f"Extracted {len(mapping)} ChEBI ↔ KEGG mappings")

# --- Export to CSV ---
#with open("chebi_kegg.csv", "w", newline="") as f:
#    writer = csv.writer(f)
#    writer.writerow(["ChEBI_ID", "KEGG_COMPOUND_IDs"])
#    for chebi, kegg_list in mapping.items():
#        writer.writerow([chebi, ";".join(kegg_list)])

# --- Export to JSON ---
with open("chebi_kegg.json", "w") as f:
    json.dump(mapping, f, indent=2)

# --- Export to text ---
#with open("chebi_kegg.txt", "w") as f:
#    for chebi, kegg_list in mapping.items():
#        f.write(f"{chebi}: {', '.join(kegg_list)}\n")

Extracted 16957 ChEBI ↔ KEGG mappings


In [3]:
import lzma
import pickle
with lzma.open("chebi_to_kegg_map.lzma", "wb") as f:
    pickle.dump(mapping, f)

## Create a dict for each KEGG reaction

In [ ]:
INPUT_DIR = "../../kegg/reactions"
OUTPUT_FILE = "./parsed_kegg_reactions.json"
parsed_reactions = []

def parse_kegg_flat_file(text):
    entry = {}
    current_key = None
    for line in text.splitlines():
        if line[:12].strip():  # new field
            current_key = line[:12].strip()
            entry[current_key] = line[12:].strip()
        elif current_key:
            entry[current_key] += " " + line[12:].strip()
    return entry

def parse_reaction_equation(rxn_str):
    # Assume directional info is unreliable
    if "=>" in rxn_str or "->" in rxn_str:
        lhs, rhs = re.split(r"=>|->", rxn_str)
    else:
        return [], []

    def parse_side(side):
        return [s.strip().split()[-1] for s in side.strip().split("+")]

    reactants = parse_side(lhs)
    products = parse_side(rhs)
    return reactants, products

# Process each KEGG reaction file
for filepath in glob.glob(os.path.join(INPUT_DIR, "reaction_R*.txt")):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    raw = parse_kegg_flat_file(content)

    reaction_id = os.path.basename(filepath).replace("reaction_", "").replace(".txt", "")

    substrates, products = parse_reaction_equation(raw.get("EQUATION", ""))

    record = {
        "reaction_id": f"{reaction_id}",
        "name": raw.get("NAME", ""),
        "ec_numbers": raw.get("ENZYME", "").split(),
        "substrates": substrates,
        "products": products,
        "pathways": [p.strip().split()[0] for p in raw.get("PATHWAY", "").split("map") if p.strip()] if "PATHWAY" in raw else [],
        "raw_equation": raw.get("EQUATION", ""),
    }

    parsed_reactions.append(record)

# Write to JSON
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(parsed_reactions, f, indent=2)

print(f"✅ Parsed {len(parsed_reactions)} reactions → {OUTPUT_FILE}")


✅ Parsed 12304 reactions → ./parsed_kegg_reactions.json


In [ ]:
with open('parsed_kegg_reactions.lzma', 'r', encoding='utf-8') as infile:
    data = json.load(infile)

import pickle 

with lzma.open('parsed_kegg_reactions-pickle.lzma', "wb") as f:
    pickle.dump(data, f)

In [13]:
import json, lzma

with lzma.open('parsed_kegg_reactions.lzma', 'rt', encoding='utf-8') as f:
    parsed_kegg_reactions = json.load(f)

In [15]:
with lzma.open('parsed_kegg_reactions_pickle.lzma', "wb") as f:
    pickle.dump(parsed_kegg_reactions, f)

In [ ]:
import lzma
import pickle
with lzma.open('parsed_kegg_reactions.lzma', 'rb') as f:
    parsed_kegg_reactions = pickle.load(f)

In [4]:
parsed_kegg_reactions

[{'reaction_id': 'R00001',
  'name': 'polyphosphate polyphosphohydrolase',
  'ec_numbers': ['3.6.1.10'],
  'direction': 'reversible',
  'substrates': ['C00404', 'C00001'],
  'products': ['C02174'],
  'pathways': [],
  'raw_equation': 'C00404 + n C00001 <=> (n+1) C02174'},
 {'reaction_id': 'R00002',
  'name': 'reduced ferredoxin:dinitrogen oxidoreductase (ATP-hydrolysing)',
  'ec_numbers': ['1.18.6.1'],
  'direction': 'reversible',
  'substrates': ['C00002', 'C00001', 'C00138'],
  'products': ['C05359', 'C00009', 'C00008', 'C00139'],
  'pathways': [],
  'raw_equation': '16 C00002 + 16 C00001 + 8 C00138 <=> 8 C05359 + 16 C00009 + 16 C00008 + 8 C00139'},
 {'reaction_id': 'R00004',
  'name': 'diphosphate phosphohydrolase; pyrophosphate phosphohydrolase',
  'ec_numbers': ['3.6.1.1'],
  'direction': 'reversible',
  'substrates': ['C00013', 'C00001'],
  'products': ['C00009'],
  'pathways': [],
  'raw_equation': 'C00013 + C00001 <=> 2 C00009'},
 {'reaction_id': 'R00005',
  'name': 'urea-1-car

In [19]:
parsed_kegg_reactions

[{'reaction_id': 'R00001',
  'name': 'polyphosphate polyphosphohydrolase',
  'ec_numbers': ['3.6.1.10'],
  'direction': 'reversible',
  'substrates': ['C00404', 'C00001'],
  'products': ['C02174'],
  'pathways': [],
  'raw_equation': 'C00404 + n C00001 <=> (n+1) C02174'},
 {'reaction_id': 'R00002',
  'name': 'reduced ferredoxin:dinitrogen oxidoreductase (ATP-hydrolysing)',
  'ec_numbers': ['1.18.6.1'],
  'direction': 'reversible',
  'substrates': ['C00002', 'C00001', 'C00138'],
  'products': ['C05359', 'C00009', 'C00008', 'C00139'],
  'pathways': [],
  'raw_equation': '16 C00002 + 16 C00001 + 8 C00138 <=> 8 C05359 + 16 C00009 + 16 C00008 + 8 C00139'},
 {'reaction_id': 'R00004',
  'name': 'diphosphate phosphohydrolase; pyrophosphate phosphohydrolase',
  'ec_numbers': ['3.6.1.1'],
  'direction': 'reversible',
  'substrates': ['C00013', 'C00001'],
  'products': ['C00009'],
  'pathways': [],
  'raw_equation': 'C00013 + C00001 <=> 2 C00009'},
 {'reaction_id': 'R00005',
  'name': 'urea-1-car

In [22]:
substrates_to_find = {'C00404', 'C00001'}

matches = [
    rxn['reaction_id']
    for rxn in parsed_kegg_reactions
    if substrates_to_find.issubset(set(rxn['substrates']))
]
matches

['R00001', 'R03042']

## Making specific dictionaries

In [25]:
import lzma
import json
import pickle

with lzma.open('parsed_kegg_reactions.lzma', 'rt', encoding='utf-8') as f:
    parsed_kegg_reactions = json.load(f)

reaction_name_dict = {
    entry['name']: entry['reaction_id']
    for entry in parsed_kegg_reactions
}

with lzma.open('reactionnames2kegg.lzma', "wb") as f:
    pickle.dump(reaction_name_dict, f)

In [26]:
substrates_dict = {
    entry['reaction_id']: entry['substrates']
    for entry in parsed_kegg_reactions
}

with lzma.open('reaction_substrates.lzma', "wb") as f:
    pickle.dump(reaction_name_dict, f)

In [27]:
products_dict = {
    entry['reaction_id']: entry['products']
    for entry in parsed_kegg_reactions
}

with lzma.open('reaction_products.lzma', "wb") as f:
    pickle.dump(reaction_name_dict, f)

In [ ]:
with lzma.open('parsed_kegg_reactions.lzma', 'rt', encoding='utf-8') as f:
    parsed_kegg_reactions = json.load(f)

reaction_name_dict = {
    entry['reaction_id']: entry['ec_numbers']
    for entry in parsed_kegg_reactions
}

with lzma.open('kegg2ec.lzma', "wb") as f:
    pickle.dump(reaction_name_dict, f)